# Train Merchant Agent on ChargebackOps

End-to-end GRPO training skeleton for the merchant-side chargeback agent.

- Environment: `ChargebackOpsEnvironment` (multi-round adversarial Issuer, arbitration ROI).
- Text interface: `training.env_adapter` (prompt build, completion parse).
- Reward: `training.reward_adapter.compute_reward` — returns the normalised episode score in `[0, 1]`.
- Trainer: `trl.GRPOTrainer` on a small base model so this fits a free Colab T4.

The first run is intentionally tiny (1 step, micro-batch of 2). Once the wiring is green, bump `max_steps` to 200 for the real curve.

## 1. Colab setup

Installs TRL, transformers, and the ChargebackOps package itself. Skip if the environment already has them.

In [ ]:
%%capture
import sys
if 'google.colab' in sys.modules:
    !pip install --quiet trl==0.11.4 transformers==4.44.2 accelerate==0.33.0 peft==0.12.0 bitsandbytes==0.43.3
    !git clone https://github.com/example/chargebackops.git /content/chargebackops
    %cd /content/chargebackops
    !pip install --quiet -e .

## 2. Sanity-check the env adapter

Run one scripted episode via the text adapter to confirm prompts render and rewards land inside `[0, 1]`.

In [ ]:
from training.env_adapter import build_prompt
from training.reward_adapter import run_episode_with_text_policy

def heuristic_text_policy(prompt: str) -> str:
    # Force the fallback path so the scripted heuristic drives the episode.
    return ''

result = run_episode_with_text_policy('goods_not_received_easy', heuristic_text_policy)
print('score', result.score, 'steps', result.steps_used, 'invalid', result.invalid_actions)

## 3. Load a small base model

`Qwen/Qwen2.5-0.5B-Instruct` fits on a free T4 with LoRA adapters. Swap in a bigger instruct model if you have the memory budget.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)

## 4. Build the training prompt dataset

GRPO expects a list of prompts; it generates K completions per prompt internally and scores each with `compute_reward`. We sample prompts from fresh environment resets across the headline catalog.

In [ ]:
from datasets import Dataset
from scenarios.simulation import list_tasks
from server.chargeback_ops_environment import ChargebackOpsEnvironment
from training.env_adapter import build_prompt

def sample_prompts(n: int = 32):
    tasks = list_tasks()
    prompts, task_ids = [], []
    for i in range(n):
        task = tasks[i % len(tasks)]
        env = ChargebackOpsEnvironment()
        obs = env.reset(task_id=task.task_id).model_dump()
        prompts.append(build_prompt(obs))
        task_ids.append(task.task_id)
    return Dataset.from_dict({'prompt': prompts, 'task_id': task_ids})

train_dataset = sample_prompts(32)
len(train_dataset)

## 5. GRPO training step

Starts with `max_steps=1` — just verify the gradient path closes. Increase to 200 for the real curve.

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from training.reward_adapter import compute_reward

def reward_fn(prompts, completions, **kwargs):
    task_ids = kwargs.get('task_id') or kwargs.get('task_ids')
    return compute_reward(prompts, completions, task_ids=task_ids)

config = GRPOConfig(
    output_dir='./grpo-merchant-agent',
    per_device_train_batch_size=2,
    num_generations=4,
    max_prompt_length=1024,
    max_completion_length=128,
    learning_rate=5e-6,
    max_steps=1,
    logging_steps=1,
    save_steps=50,
    gradient_accumulation_steps=1,
    bf16=torch.cuda.is_available(),
    report_to='none',
)
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_fn],
    args=config,
    train_dataset=train_dataset,
)
trainer.train()

## 6. Evaluate the trained policy

Runs one rollout per headline task with the trained model as the text policy and reports the per-task scores plus the overall mean.

In [ ]:
import torch
from statistics import mean

def model_text_policy(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, do_sample=False, temperature=0.0, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

scores = []
for task in list_tasks():
    result = run_episode_with_text_policy(task.task_id, model_text_policy)
    scores.append(result.score)
    print(f'{task.task_id:32s} score={result.score:.4f}  steps={result.steps_used}  invalid={result.invalid_actions}')
print('mean score', mean(scores))

## Next steps

1. Bump `max_steps` in step 5 to 200 (save checkpoints at 0/50/100/150/200).
2. Record per-checkpoint mean score and plot the curve.
3. Compare against the fixed-policy baselines in `runners/benchmark_runner.py`.
